In [1]:
import wandb
import pandas as pd
from scipy import stats
import json

In [2]:
api = wandb.Api()

In [3]:
# Define entity and project name ("plp-ml-20m" or "plp-coveo-search" or "plp-coveo-pageview")
entity, project = "your_entity", "plp-ml-20m" 

# Define artifact folder
artifact_folder = "./saved/run_artifacts/"
# Fetch runs
runs = api.runs(entity + "/" + project)
if not runs:
    raise ValueError(f"No runs found in project {project} under entity {entity}")
table_key = "per_user_metrics"
results = {}


In [4]:
# Fetch artifacts
for run in runs:
    if table_key in run.summary:
        run_id = run.id
        seed = run.config["external_config_dict"]["seed"]
        try:
            artifact = api.artifact(f"{entity}/{project}/run-{run_id}-per_user_metrics:latest")
            table_file_path = f"{artifact_folder}{run.name}/{seed}/per_user_metrics.table.json"
            artifact_dir = artifact.download(root=f"{artifact_folder}/{run.name}/{seed}")
            # Load JSON
            with open(table_file_path, "r") as f:
                table_data = json.load(f)
                df = pd.DataFrame(table_data["data"], columns=table_data["columns"])
                results[run.name] = {}
                results[run.name][seed] = df
        except wandb.CommError as e:
            print(f"Error fetching artifact: {e}")

wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downlo

In [5]:
#Extract data
results = {}
for run in runs:
    if table_key in run.summary:
        run_id = run.id
        seed = run.config["external_config_dict"]["seed"]
        table_file_path = f"{artifact_folder}{run.name}/{seed}/per_user_metrics.table.json"
        with open(table_file_path, "r") as f:
            table_data = json.load(f)
            df = pd.DataFrame(table_data["data"], columns=table_data["columns"])
            if results.get(run.name) is None:
                results[run.name] = {}
            results[run.name][seed] = df

In [6]:
results.keys()

dict_keys(['ml-20m-first-genre_id-gru4rec', 'ml-20m-first-genre_pe-narm', 'ml-20m-first-genre_pe-bert4rec', 'ml-20m-first-genre_id-sasrec', 'ml-20m-extended-genre_pe-bert4rec', 'ml-20m-extended-genre_id-bert4rec', 'ml-20m-first-genre_id-nextitnet', 'ml-20m-extended-genre_pe-caser', 'ml-20m-extended-genre_pe-lightsans', 'ml-20m-first-genre_pe-caser', 'ml-20m-first-genre_id-narm', 'ml-20m-extended-genre_pe-narm', 'ml-20m-first-genre_pe-gru4rec', 'ml-20m-first-genre_pe-lightsans', 'ml-20m-first-genre_pe-core', 'ml-20m-first-genre_pe-sasrec', 'ml-20m-first-genre_id-bert4rec', 'ml-20m-first-genre_pe-nextitnet', 'ml-20m-extended-genre_id-lightsans', 'ml-20m-first-genre_id-core', 'ml-20m-first-genre_id-lightsans', 'ml-20m-first-genre_id-caser', 'ml-20m-extended-genre_pe-nextitnet', 'ml-20m-extended-genre_id-narm', 'ml-20m-extended-genre_id-nextitnet', 'ml-20m-extended-genre_pe-gru4rec', 'ml-20m-extended-genre_id-sasrec', 'ml-20m-extended-genre_id-caser', 'ml-20m-extended-genre_pe-core', 'ml-2

In [8]:
models = ['bert4rec', 'caser','gru4rec', 'narm', 'sasrec', 'core', 'lightsans', 'nextitnet']

if project == "plp-ml-20m":
    base_prefix ="ml-20m-title_id-"
    prefixes = ['ml-20m-extended-genre_id-', 'ml-20m-extended-genre_pe-', 'ml-20m-first-genre_id-', 'ml-20m-first-genre_pe-']
if project == "plp-coveo-search":
    base_prefix = 'coveo-sl-title_id-'
    prefixes = ['coveo-sl-search-category_id-','coveo-sl-search-category_pe-','coveo-sl-search-firstcategory_id-','coveo-sl-search-firstcategory_pe-',
    'coveo-sl-search-firstitem_id-', 'coveo-sl-search-query_pe-']
if project == "plp-coveo-pageview":
    base_prefix = 'coveo-title_id-'
    prefixes = ['coveo-pageview-title_id-']

for model in models:
    print("\nModel: ", model)
    for metric in ['recall@1', 'recall@5', 'recall@10', 'ndcg@5', 'ndcg@10']:
        p_values = {}
        print("Metric: ", metric)
        for prefix in prefixes:
            baseline = base_prefix + model
            target = prefix + model
        
            result_baseline = results[baseline]
            result_target = results[target]
        
            #Concat all seeds
            result_baseline = pd.concat(result_baseline.values())
            result_target =  pd.concat(result_target.values())
            #Perform t-test
            p_value = stats.ttest_rel(result_baseline[metric], result_target[metric], alternative="less")
            p_values[metric] = p_value
    
            significant_metrics = len([key for key, value in p_values.items() if value.pvalue < 0.01])>0
            print("Significant metrics for p <0.01: ", prefix, significant_metrics)
            significant_metrics = len([key for key, value in p_values.items() if value.pvalue < 0.05])>0
            print("Significant metrics for p<0.05: ", prefix, significant_metrics)
            print("---------------------------")
    
   


Model:  bert4rec
Metric:  recall@1
Significant metrics for p <0.01:  ml-20m-extended-genre_id- True
Significant metrics for p<0.05:  ml-20m-extended-genre_id- True
---------------------------
Significant metrics for p <0.01:  ml-20m-extended-genre_pe- True
Significant metrics for p<0.05:  ml-20m-extended-genre_pe- True
---------------------------
Significant metrics for p <0.01:  ml-20m-first-genre_id- True
Significant metrics for p<0.05:  ml-20m-first-genre_id- True
---------------------------
Significant metrics for p <0.01:  ml-20m-first-genre_pe- True
Significant metrics for p<0.05:  ml-20m-first-genre_pe- True
---------------------------
Metric:  recall@5
Significant metrics for p <0.01:  ml-20m-extended-genre_id- True
Significant metrics for p<0.05:  ml-20m-extended-genre_id- True
---------------------------
Significant metrics for p <0.01:  ml-20m-extended-genre_pe- True
Significant metrics for p<0.05:  ml-20m-extended-genre_pe- True
---------------------------
Significant metr